In [1]:
import pandas as pd
import numpy as np
import random
from itertools import product

# 设置随机种子以确保可重复性
random.seed(42)
np.random.seed(42)

def create_negative_samples(positive_df, gene_list=None, allow_self_loop=False):
    """
    为基因调控网络构建负样本
    
    参数:
    positive_df: pandas DataFrame, 正样本数据，至少包含两列表示基因对
    gene_list: list, 所有基因的列表（可选）。如果为None，则从正样本中提取所有基因
    allow_self_loop: bool, 是否允许自环（基因对自身）。默认为False
    
    返回:
    negative_df: pandas DataFrame, 负样本数据
    """
    
    # 确保正样本数据有两列
    if len(positive_df.columns) < 2:
        raise ValueError("正样本DataFrame至少需要两列来表示基因对")
    
    # 获取基因对列名
    gene_col1, gene_col2 = positive_df.columns[:2]
    
    # 如果没有提供基因列表，从正样本中提取所有唯一基因
    if gene_list is None:
        tf_list = list(set(positive_df[gene_col1].tolist()))
        gene_list = list(set((positive_df[gene_col2].tolist())))

    # 创建所有可能的基因对（包括自环或不包括，根据参数）
    if allow_self_loop:
        all_possible_pairs = list(product(gene_list, repeat=2))
    else:
        all_possible_pairs = [(g1, g2) for g1 in tf_list for g2 in gene_list if g1 != g2]
    
    # 将正样本转换为集合以便快速查找
    positive_pairs = set(positive_df.apply(lambda row: tuple(row[:2]), axis=1))
    
    # 获取所有非正样本的基因对
    negative_candidates = [pair for pair in all_possible_pairs if pair not in positive_pairs]
    
    # 检查是否有足够的负样本候选
    n_positive = len(positive_df)
    if len(negative_candidates) < n_positive:
        print(f"警告: 负样本候选数量({len(negative_candidates)})少于正样本数量({n_positive})")
        print("将使用所有可用的负样本候选")
        n_samples = min(n_positive, len(negative_candidates))
    else:
        n_samples = n_positive
    
    # 随机抽取负样本
    selected_negative_pairs = random.sample(negative_candidates, n_samples)
    
    # 创建负样本DataFrame
    negative_df = pd.DataFrame(selected_negative_pairs, columns=[gene_col1, gene_col2])
    
    # 如果正样本有其他列，可以在负样本中添加对应的占位符列
    if len(positive_df.columns) > 2:
        for col in positive_df.columns[2:]:
            # 对于数值列，可以填充0；对于分类列，可以填充空值或特定值
            if pd.api.types.is_numeric_dtype(positive_df[col]):
                negative_df[col] = 0
            else:
                negative_df[col] = np.nan
    
    return negative_df

# 示例用法
if __name__ == "__main__":   
    
    pos = pd.read_csv("BL--network.csv")
    #del pos["Score"]
    print(pos)
    pos["label"] = 1
    pos.columns=["TF","Target","label"]
    # 方法1: 使用从正样本中提取的基因列表
    negative_df1 = create_negative_samples(pos)
    negative_df1["label"]=0
    negative_df1.columns=["TF","Target","label"]
    print(len(set(list(negative_df1["TF"]))))
    print(len(set(list(pos["TF"]))))
    print((set(list(pos["TF"]))))
    all=pd.concat([pos,negative_df1])   
    print(all)
    all.to_csv("AllPair.csv",index=False)

     Gene1    Gene2
0     AATF   CDKN1A
1     AATF      MYC
2     AATF     TP53
3      AHR    BRCA1
4      AHR    CCND1
...    ...      ...
5346  TP53    SULF2
5347  TP53     SVIL
5348  TP53  TBC1D31
5349  TP53    UQCC1
5350  TP53     UTRN

[5351 rows x 2 columns]
332
332
{'TCEB3', 'TAF7', 'XRCC6', 'CEBPA', 'KAT7', 'POLR2D', 'PIAS3', 'FOXH1', 'CSRP2', 'NANOG', 'MLXIPL', 'MCM3', 'JUN', 'NFATC2', 'ETV1', 'FOXA2', 'ILF2', 'TRIP13', 'MYCBP', 'HNRNPK', 'SRF', 'COPS5', 'NCOA6', 'FOSB', 'POLR1D', 'POLR2K', 'TCEA1', 'TCF3', 'AES', 'ING1', 'CHD8', 'HMGA2', 'THRB', 'TOP2B', 'AEBP1', 'HIST1H1C', 'POLR2G', 'HMGN2', 'SUV39H2', 'SP100', 'E2F5', 'SOX2', 'ILF3', 'FHL2', 'CHAF1A', 'RBPJ', 'TFE3', 'ZRANB2', 'LIN28A', 'TCF7L1', 'HNRNPD', 'DNMT3A', 'BDP1', 'TOPORS', 'RNF19A', 'NR1I3', 'BCL11B', 'SSRP1', 'TRIM27', 'GTF2I', 'ENO1', 'MEF2C', 'HMGN1', 'HIF1AN', 'RNF2', 'CREB1', 'ID3', 'PRDM1', 'POLR3D', 'CBX3', 'BTF3', 'ETS1', 'SRSF2', 'PITX2', 'IVNS1ABP', 'EPAS1', 'ZNRD1', 'SALL2', 'NR3C1', 'CREB3L3', 'NUPR1

In [2]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
a

,BMPR1A,P4HA2,HNRNPUL1,BBS9,BIRC5,ESYT1,ZNF101,ACOX2,B4GALT3,BRCA2,...,CUL2,POLR1D,BTG1,ZNF143,FXYD5,CEBPZ,ZNF33A,ETV4,EXOSC3,RAB34
0,1.445663,5.051255,4.601554,7.150692,5.291629,4.277672,3.911356,0.000000,0.000000,0.207793,...,4.597495,2.947668,2.187432,4.256996,1.898483,4.288373,6.171631,0.846265,6.366060,5.582670
1,3.935724,0.000000,3.466784,6.488758,5.955734,3.131993,0.000000,0.000000,0.000000,5.213946,...,0.000000,5.334883,0.000000,7.870291,2.125172,5.902701,5.051729,0.000000,5.604807,5.232097
2,0.000000,0.000000,1.570560,0.944416,3.935733,3.745280,0.000000,0.000000,0.000000,4.994236,...,4.163249,6.063721,3.698652,0.000000,4.726221,3.332651,1.581901,2.692360,6.697732,5.176315
3,3.726232,0.775666,2.567487,5.142511,3.452846,6.121942,6.540570,0.000000,5.422771,3.916983,...,4.729063,5.803100,2.948491,5.532220,2.195316,4.088769,4.474748,0.000000,6.035934,6.142789
4,0.000000,4.267716,4.415982,2.392295,6.890179,0.000000,0.000000,0.000000,0.000000,4.469756,...,0.000000,7.174356,4.896345,7.497876,0.000000,3.739643,0.000000,3.760210,0.000000,6.218894
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,0.000000,0.000000,0.767646,5.797324,0.000000,0.000000,0.000000,4.042163,0.000000,0.000000,...,5.335637,2.675247,0.772274,2.293632,0.000000,0.000000,0.000000,0.000000,0.000000,5.535241
421,0.000000,0.000000,0.000000,5.837595,0.000000,6.522197,0.000000,7.067800,5.345925,0.000000,...,0.000000,0.000000,0.000000,7.763299,0.000000,0.000000,0.000000,0.000000,6.379933,7.763060
422,1.320433,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,5.783310,0.000000,0.000000,0.000000,0.000000
423,0.349682,5.957485,0.954636,7.834661,0.000000,0.000000,0.000000,0.000000,3.763443,0.000000,...,5.086945,5.774439,4.855751,3.881870,0.000000,3.969565,3.329870,0.000000,0.000000,6.569676


In [3]:
#转为大写
import pandas as pd
a=pd.read_csv("pathway_hHEP.csv")
col = list(a.columns)
cols = [i.upper() for i in col]
a.columns=cols
a.to_csv("pathway_hHEP.csv",index=False)

In [4]:
import pandas as pd
a=pd.read_csv("Target.csv")
column_names = list(a["Target"])
uppercase_names = [name.upper() for name in column_names]
a["Target"] = uppercase_names
a.to_csv("AllPair.csv",index=False)

KeyError: 'Target'

In [5]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
column_names = list(a.columns)
uppercase_names = [name.upper() for name in column_names]
a.columns = uppercase_names
a.to_csv("ExpressionData.csv",index=False)